In [21]:
import torch
import torch.nn.functional as F
import os
import sys
from tqdm import tqdm
project_root = '/public/home/zhangshikang/project/decouple_detr/DecoupleDETR/'
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)
from dataset import (build_dataset, build_loader)
from models import build_model
from util.config import ConfigDict
from util.distributed import init_distributed_mode
from trainer.supervised_trainer.engine import trainer

- we measure the flop of our model and comparing with other models

In [3]:
cfg = ConfigDict.from_file('config/experiments/pannuke_supervised.yaml')
init_distributed_mode(cfg)
model_trainer = trainer(cfg,False)

Not using distributed mode


/public/home/zhangshikang/.conda/envs/detr/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/public/home/zhangshikang/.conda/envs/detr/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


loading annotations into memory...
Done (t=0.88s)
creating index...
index created!


In [6]:
from fvcore.nn import FlopCountAnalysis

In [7]:
def count_flops(model, input_tuple):
    model.eval()
    flops = FlopCountAnalysis(model, input_tuple)
    return flops.total()

In [15]:
imgs.tensors.device

device(type='cpu')

In [25]:
model = model_trainer.model.to('cuda')
test_loader = model_trainer.test_loader
test_tqdm = tqdm(test_loader,)
flops = []
model.eval()
with torch.no_grad():
    for img,targets in test_tqdm:
        img = img.tensors.to('cuda')
        targets = [{k: v.to('cuda') if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]
        flop = count_flops(model, (img,targets,'train'))
        flops.append(flop)


  0%|          | 0/316 [00:00<?, ?it/s]Unsupported operator aten::fill_ encountered 16 time(s)
Unsupported operator aten::add encountered 192 time(s)
Unsupported operator aten::rsqrt encountered 53 time(s)
Unsupported operator aten::mul encountered 223 time(s)
Unsupported operator aten::sub encountered 61 time(s)
Unsupported operator aten::max_pool2d encountered 1 time(s)
Unsupported operator aten::add_ encountered 19 time(s)
Unsupported operator aten::cumsum encountered 9 time(s)
Unsupported operator aten::div encountered 60 time(s)
Unsupported operator aten::pow encountered 8 time(s)
Unsupported operator aten::sin encountered 8 time(s)
Unsupported operator aten::cos encountered 8 time(s)
Unsupported operator aten::prod encountered 1 time(s)
Unsupported operator aten::sum encountered 8 time(s)
Unsupported operator aten::linspace encountered 8 time(s)
Unsupported operator aten::meshgrid encountered 4 time(s)
Unsupported operator aten::softmax encountered 17 time(s)
Unsupported operator

In [26]:
flops

[[...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],
 [...],


In [27]:
mean(flops)

NameError: name 'mean' is not defined